# Data Speaks!! — BERT Fine-Tuning for Rating Regression

This notebook fine-tunes **`bert-base-uncased`** on the IMDB movie review dataset for **Continuous Rating Regression** (predicting a score from 1 to 10).

### Pipeline Steps
| Step | Description |
|------|-------------|
| 0 | Environment & directory setup |
| 1 | Load & verify raw data (25K train + 25K test) |
| 2 | Data audit — nulls, duplicates, text stats, HTML artifacts |
| 3 | Text cleaning — HTML strip, whitespace normalize, dedup |
| 4 | Stratified train/validation split (90/10, seed=42) |
| 5 | BERT tokenization & PyTorch DataLoaders |
| 6 | BERT model training (3 epochs, GPU) & test-set evaluation (MSE / MAE) |
| 7 | Export metrics CSV |

All artifacts are saved under **`artifacts/`**.


---
## Step 0 — Environment & Directory Setup

In [1]:
import os
import sys
import time
import re
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

# Reproducibility 
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

# Paths
DATA_DIR    = os.path.join("..", "..", "processed_for_regression")
ARTIFACTS   = os.path.join(".", "artifacts")
MODELS_DIR  = os.path.join(ARTIFACTS, "models")
METRICS_DIR = os.path.join(ARTIFACTS, "metrics")
PLOTS_DIR   = os.path.join(ARTIFACTS, "plots")

for d in [ARTIFACTS, MODELS_DIR, METRICS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {vram_gb:.1f} GB")
else:
    print("No GPU detected — training will be very slow on CPU.")

c:\Users\dhanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device  : cuda
GPU     : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM    : 8.5 GB


---
## Step 1 — Acquire and Verify Data

In [2]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "imdb_regression_train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "imdb_regression_test.csv"))

print(f"Train shape   : {train_df.shape}")
print(f"Test  shape   : {test_df.shape}")
print(f"Columns       : {list(train_df.columns)}")

print(f"\nTrain rating counts:\n{train_df['rating'].value_counts().sort_index()}")
print(f"\nTest  rating counts:\n{test_df['rating'].value_counts().sort_index()}")


Train shape   : (25000, 2)
Test  shape   : (25000, 2)
Columns       : ['text', 'rating']

Train rating counts:
rating
1     5100
2     2284
3     2420
4     2696
7     2496
8     3009
9     2263
10    4732
Name: count, dtype: int64

Test  rating counts:
rating
1     5022
2     2302
3     2541
4     2635
7     2307
8     2850
9     2344
10    4999
Name: count, dtype: int64


---
## Step 2 — Data Audit (Read-Only)

In [3]:
#Null 
print("Counting null values")
print(f"Train nulls:\n{train_df.isnull().sum()}")
print(f"\nTest nulls:\n{test_df.isnull().sum()}")

#Duplicate
train_dups = train_df.duplicated(subset=["text"]).sum()
test_dups  = test_df.duplicated(subset=["text"]).sum()
print(f"\nCounting Duplicates")
print(f"Train: {train_dups}  |  Test: {test_dups}")

# Text length
train_len = train_df["text"].str.len()
test_len  = test_df["text"].str.len()
print(f"\nStatistics of character length")
print(f"Train : min={train_len.min()}, max={train_len.max()}, "
      f"mean={train_len.mean():.1f}, median={train_len.median():.1f}")
print(f"Test  : min={test_len.min()}, max={test_len.max()}, "
      f"mean={test_len.mean():.1f}, median={test_len.median():.1f}")

# HTML
html_train = train_df["text"].str.contains("<br", regex=False).sum()
html_test  = test_df["text"].str.contains("<br", regex=False).sum()
print(f"\nChecking html tags")
print(f"Train: {html_train} ({html_train/len(train_df):.1%})")
print(f"Test:  {html_test}  ({html_test/len(test_df):.1%})")


Counting null values
Train nulls:
text      0
rating    0
dtype: int64

Test nulls:
text      0
rating    0
dtype: int64

Counting Duplicates
Train: 96  |  Test: 199

Statistics of character length
Train : min=52, max=13704, mean=1325.1, median=979.0
Test  : min=32, max=12988, mean=1293.8, median=962.0

Checking html tags
Train: 14665 (58.7%)
Test:  14535  (58.1%)


---
## Step 3 — Clean the Text

In [4]:
def clean_text(text: str) -> str:
    """Strip HTML tags/entities, normalize whitespace."""
    text = html.unescape(text)                                    # decode &amp; etc.
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)  # <br /> → space
    text = re.sub(r"<[^>]+>", " ", text)                         # any remaining tags
    text = re.sub(r"\s+", " ", text).strip()                     # collapse whitespace
    return text

train_df["text_clean"] = train_df["text"].apply(clean_text)
test_df["text_clean"]  = test_df["text"].apply(clean_text)

# Remove exact duplicates (keep first occurrence)
train_clean = (
    train_df.drop_duplicates(subset=["text_clean"], keep="first")
    .copy()[["text_clean", "rating"]]
    .rename(columns={"text_clean": "text"})
)
test_clean = (
    test_df.drop_duplicates(subset=["text_clean"], keep="first")
    .copy()[["text_clean", "rating"]]
    .rename(columns={"text_clean": "text"})
)

print(f"Train rows: {len(train_df)} → {len(train_clean)} (removed {len(train_df)-len(train_clean)} duplicates)")
print(f"Test  rows: {len(test_df)}  → {len(test_clean)}  (removed {len(test_df)-len(test_clean)} duplicates)")

# Save cleaned datasets
train_clean.to_csv(os.path.join(ARTIFACTS, "bert_imdb_regression_train_clean.csv"), index=False)
test_clean.to_csv(os.path.join(ARTIFACTS, "bert_imdb_regression_test_clean.csv"), index=False)
print(f"Saved cleaned CSVs to {ARTIFACTS}/")


Train rows: 25000 → 24902 (removed 98 duplicates)
Test  rows: 25000  → 24799  (removed 201 duplicates)
Saved cleaned CSVs to .\artifacts/


---
## Step 4 — Train / Validation Split

In [5]:
# For regression, we can still stratify by the discrete rating values since they are integers from 1-10.
train_split, val_split = train_test_split(
    train_clean,
    test_size=0.10,
    random_state=SEED,
    stratify=train_clean["rating"],
)

print(f"Train subset : {len(train_split):>6}  ({len(train_split)/len(train_clean):.1%})")
print(f"Val   subset : {len(val_split):>6}  ({len(val_split)/len(train_clean):.1%})")
print(f"Test  (held) : {len(test_clean):>6}")


Train subset :  22411  (90.0%)
Val   subset :   2491  (10.0%)
Test  (held) :  24799


---
## Step 5 — BERT Tokenization & DataLoaders

In [6]:
MODEL_NAME = "bert-base-uncased"
tokenizer  = BertTokenizerFast.from_pretrained(MODEL_NAME)

class IMDBRegressionDataset(Dataset):
    """PyTorch Dataset that tokenizes on-the-fly. For Regression, labels must be float."""

    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        # Important: Use float for regression
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

BATCH_SIZE = 16

train_ds = IMDBRegressionDataset(train_split["text"].values, train_split["rating"].values, tokenizer)
val_ds   = IMDBRegressionDataset(val_split["text"].values,   val_split["rating"].values,   tokenizer)
test_ds  = IMDBRegressionDataset(test_clean["text"].values,  test_clean["rating"].values,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Tokenizer vocab : {tokenizer.vocab_size}")
print(f"Dataset sizes   : train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Batches         : train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")


Tokenizer vocab : 30522
Dataset sizes   : train=22411, val=2491, test=24799
Batch size      : 16
Batches         : train=1401, val=156, test=1550


---
## Step 6 — Train BERT (3 Epochs, Mixed Precision)

In [7]:
# Note: num_labels=1 tells HuggingFace to use MSELoss for regression
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
model.to(device)

EPOCHS      = 3
LR          = 2e-5
total_steps = len(train_loader) * EPOCHS

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

history = {"train_loss": [], "val_loss": [], "val_mae": []}

print(f"Training {MODEL_NAME} for {EPOCHS} epochs on {device}…")
print(f"Total optimiser steps: {total_steps}")
t0 = time.time()

for epoch in range(EPOCHS):
    t_epoch = time.time()

    # Training
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            out  = model(ids, attention_mask=mask, labels=lbls)
            loss = out.loss # This is MSELoss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()

        if step % 300 == 0 or step == len(train_loader):
            print(f"  Epoch {epoch+1}/{EPOCHS}  step {step}/{len(train_loader)}  "
                  f"batch_loss(MSE)={loss.item():.4f}")

    avg_train = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss_sum = 0.0
    vp, vt = [], []

    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                out = model(ids, attention_mask=mask, labels=lbls)

            val_loss_sum += out.loss.item()
            # For regression, logits is shape (batch_size, 1)
            vp.extend(out.logits.squeeze(1).cpu().numpy())
            vt.extend(lbls.cpu().numpy())

    avg_val = val_loss_sum / len(val_loader)
    v_mae   = mean_absolute_error(vt, vp)

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_mae"].append(v_mae)

    dt = time.time() - t_epoch
    print(f"  ── Epoch {epoch+1} done in {dt:.0f}s │ "
          f"train_loss(MSE)={avg_train:.4f}  val_loss(MSE)={avg_val:.4f}  val_MAE={v_mae:.4f}\n")

total_wall_time = time.time() - t0
print(f"Total training wall-clock: {total_wall_time:.1f}s  ({total_wall_time/60:.1f} min)")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 456.55it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Training bert-base-uncased for 3 epochs on cuda…
Total optimiser steps: 4203


C:\Users\dhanu\AppData\Local\Temp\ipykernel_722320\4124197591.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  Epoch 1/3  step 300/1401  batch_loss(MSE)=6.7280
  Epoch 1/3  step 600/1401  batch_loss(MSE)=0.9628
  Epoch 1/3  step 900/1401  batch_loss(MSE)=3.7291
  Epoch 1/3  step 1200/1401  batch_loss(MSE)=1.2048
  Epoch 1/3  step 1401/1401  batch_loss(MSE)=1.0339
  ── Epoch 1 done in 488s │ train_loss(MSE)=6.3916  val_loss(MSE)=2.5798  val_MAE=1.1139

  Epoch 2/3  step 300/1401  batch_loss(MSE)=1.0851
  Epoch 2/3  step 600/1401  batch_loss(MSE)=0.6935
  Epoch 2/3  step 900/1401  batch_loss(MSE)=1.3237
  Epoch 2/3  step 1200/1401  batch_loss(MSE)=1.3186
  Epoch 2/3  step 1401/1401  batch_loss(MSE)=1.0731
  ── Epoch 2 done in 497s │ train_loss(MSE)=1.9574  val_loss(MSE)=2.2199  val_MAE=1.0152

  Epoch 3/3  step 300/1401  batch_loss(MSE)=0.6501
  Epoch 3/3  step 600/1401  batch_loss(MSE)=0.9440
  Epoch 3/3  step 900/1401  batch_loss(MSE)=0.3559
  Epoch 3/3  step 1200/1401  batch_loss(MSE)=0.3541
  Epoch 3/3  step 1401/1401  batch_loss(MSE)=0.9024
  ── Epoch 3 done in 492s │ train_loss(MSE)=1.301

---
## Step 7 — Final Evaluation & Export Metrics

In [8]:
# Final Evaluation on the held-out TEST set
model.eval()
test_preds, test_truths = [], []

print("Running evaluation on test set...")
with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            out = model(ids, attention_mask=mask)
            
        test_preds.extend(out.logits.squeeze(1).cpu().numpy())
        test_truths.extend(lbls.cpu().numpy())

test_mse = mean_squared_error(test_truths, test_preds)
test_mae = mean_absolute_error(test_truths, test_preds)
test_r2  = r2_score(test_truths, test_preds)

print(f"\nTest MSE : {test_mse:.4f}")
print(f"Test MAE : {test_mae:.4f}")
print(f"Test R^2 : {test_r2:.4f}")

# Save metrics
metrics_df = pd.DataFrame([{
    "model": "BERT_Regression",
    "test_mse": test_mse,
    "test_mae": test_mae,
    "test_r2": test_r2
}])
metrics_df.to_csv(os.path.join(METRICS_DIR, "bert_regression_metrics.csv"), index=False)
print(f"\nSaved metrics to {METRICS_DIR}/bert_regression_metrics.csv")


Running evaluation on test set...

Test MSE : 2.2462
Test MAE : 0.9897
Test R^2 : 0.8154

Saved metrics to .\artifacts\metrics/bert_regression_metrics.csv
